In [81]:
import dagshub
import mlflow
from mlflow.tracking import MlflowClient
import joblib
import numpy as np

dagshub.init(repo_owner="adola23", repo_name="HousePricesML", mlflow=True)

# FORCE correct connection
mlflow.set_tracking_uri("https://dagshub.com/adola23/HousePricesML.mlflow")
mlflow.set_registry_uri("https://dagshub.com/adola23/HousePricesML.mlflow")

Initialized MLflow to track repo "adola23/HousePricesML"

Repository adola23/HousePricesML initialized!

# მოდელის და პრეპროცესინგის ჩამოტვირთვა

In [82]:
MODEL_NAME = "bestmodel5"
MODEL_VERSION = 2

model = mlflow.pyfunc.load_model(f"models:/{MODEL_NAME}/{MODEL_VERSION}")

In [83]:

client = MlflowClient()

model_version = client.get_model_version(
    name=MODEL_NAME,
    version=MODEL_VERSION
)

run_id = model_version.run_id

from mlflow.tracking import MlflowClient

artifacts = client.list_artifacts(run_id)

for a in artifacts:
    print(a.path)

preprocessing.pkl


In [84]:
preprocessing_path = mlflow.artifacts.download_artifacts(
    run_id=run_id,
    artifact_path="preprocessing.pkl"
)
print(preprocessing_path)
artifacts = joblib.load(preprocessing_path)

C:\Users\sdola\AppData\Local\Temp\tmpg3agxogz\preprocessing.pkl


In [85]:
num_cols = artifacts["num_cols"]
cat_cols = artifacts["cat_cols"]
train_means = artifacts["train_means"]
top_cols = artifacts["top_cols"]
ohe_dummies = artifacts["ohe_dummies"]
selected_features = artifacts["selected_features"]

In [86]:
import pandas as pd

test = pd.read_csv("data/test.csv")

# პრეპროცესინგი

In [87]:
X = test.drop(columns=["Id"])

In [89]:
# ყველაფერი ისეა გაკეთებული როგორც ექსპერიმენტში
for col in num_cols:
    if X[col].isna().sum() > 0:
        X[f"{col}_isnan"] = X[col].isna().astype(int)


for col in num_cols:
    mean_value = train_means[col]
    X[col].fillna(mean_value, inplace=True)

for col in num_cols:
    if (X[col] > 0).all():
        X[f"{col}_log"] = np.log1p(X[col])

for i in range(len(top_cols)):
    for j in range(i+1, len(top_cols)):
        c1 = top_cols[i]
        c2 = top_cols[j]
        X[f"{c1}_x_{c2}"] = X[c1] * X[c2]


for col in cat_cols:
    # one hot encoding
    dummies_train = ohe_dummies[col]
    dummies = pd.get_dummies(X[col], prefix=col)
    dummies = dummies.reindex(columns=dummies_train, fill_value=0)
    X = pd.concat([X.drop(columns=[col]), dummies], axis=1)

X = X[selected_features]

# პასუხების მიღება

In [90]:
Y_log = model.predict(X)
Y = np.expm1(Y_log)
print(Y)

[120881.26444756 159505.40396933 177283.03676799 ... 152706.27979701
 117164.44176428 207223.75666098]


In [91]:
submission = pd.DataFrame({
    "Id": test["Id"],
    "SalePrice": Y
})

submission.to_csv("submission.csv", index=False)